In [3]:
import os
import json
import glob
import psycopg2
from psycopg2 import extras
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

# Add your database connection parameters directly to your root .env file!
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "medical_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS")
DB_PORT = os.getenv("DB_PORT", "5432")

print("PostgreSQL ingestion dependencies configured.")

PostgreSQL ingestion dependencies configured.


In [5]:
# Connect directly to clean raw landing schemas inside the analytics cluster
conn = psycopg2.connect(
    host=DB_HOST, database=DB_NAME, user=DB_USER, password=DB_PASS, port=DB_PORT
)
cursor = conn.cursor()

# 1. Create a dedicated raw schema boundary to segregate raw data from transformations
cursor.execute("CREATE SCHEMA IF NOT EXISTS raw;")

# 2. Provision our structured raw storage landing engine
create_table_query = """
CREATE TABLE IF NOT EXISTS raw.telegram_messages (
    message_id VARCHAR(50),
    channel_name VARCHAR(100),
    message_date TIMESTAMP,
    message_text TEXT,
    has_media BOOLEAN,
    image_path VARCHAR(255),
    views INT,
    forwards INT,
    inserted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (channel_name, message_id)
);
"""
cursor.execute(create_table_query)
conn.commit()
print("Database 'raw.telegram_messages' ingestion table mapped successfully.")

Database 'raw.telegram_messages' ingestion table mapped successfully.


In [6]:
# Identify all json data partitions down inside our Data Lake using glob mapping paths
json_files = glob.glob("../data/raw/telegram_messages/*/*.json")
print(f"Discovered {len(json_files)} partition target components for pipeline processing.")

records_ingested = 0

for file_path in json_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        file_data = json.load(f)
        
    records_to_insert = []
    for item in file_data:
        records_to_insert.append((
            str(item["message_id"]),
            item["channel_name"],
            item["message_date"],
            item["message_text"],
            item["has_media"],
            item["image_path"],
            item["views"],
            item["forwards"]
        ))
    
    # Bulk insert using upsert logic to prevent duplicate primary key failures on re-runs
    upsert_query = """
    INSERT INTO raw.telegram_messages (message_id, channel_name, message_date, message_text, has_media, image_path, views, forwards)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (channel_name, message_id) DO UPDATE SET
        message_text = EXCLUDED.message_text,
        views = EXCLUDED.views,
        forwards = EXCLUDED.forwards;
    """
    
    extras.execute_batch(cursor, upsert_query, records_to_insert)
    records_ingested += len(records_to_insert)

conn.commit()
cursor.close()
conn.close()
print(f"Pipeline Ingestion Process Completed. Total written warehouse records: {records_ingested}")

Discovered 3 partition target components for pipeline processing.
Pipeline Ingestion Process Completed. Total written warehouse records: 276
